# Session-squeeze discovery — short & long

A trader-described 3-session setup. The **short squeeze** version:

1. **Grind (asia)** — OI rising, funding negative, price drifts down. Shorts pile up.
2. **Sweep + squeeze (lon ∪ ny)** — price takes out the grind low, rips back up toward
   the grind open; at the sweep bar, perp CVD makes a deeper low than spot CVD.
3. **Fade (NY close)** — price gives back a meaningful chunk of the squeeze.

The **long squeeze** is the mirror: longs pile up, sweep above asia high, dump
through asia open, then a partial bounce back into the close.

| | Short squeeze | Long squeeze |
|---|---|---|
| Grind | OI↑, **funding<0**, asia close<open | OI↑, **funding>0**, asia close>open |
| Sweep target | day_low < asia_low | day_high > asia_high |
| Reclaim | day_high ≥ asia_open | day_low ≤ asia_open |
| Magnitude | (day_high−day_low) ≥ 0.5×asia_range | same |
| Fade / Bounce | ny_close ≪ day_high | ny_close ≫ day_low |
| CVD divergence at sweep bar | perp_cvd < spot_cvd | perp_cvd > spot_cvd |
| London CVD direction | net buying (+) | net selling (−) |

## v3 score key

Each side has 4 legs / 10 checks: grind (3) + sweep-squeeze (4) + fade (2) +
cvd_divergence (1). A day can only fire one side (grind direction is mutually
exclusive), but a *chop* day can score 7-8 on both sides simultaneously —
those are two-way liquidation flushes.

## Data

`cd_futures_ohlcv`, `cd_spot_binance`, `cd_open_interest`, `cd_funding_rate`
(all hourly from `prod.db`). Binding window starts 2022-01-30 (when OI begins).

## Sessions (UTC)

Asia 00-07, London 07-14, NY 14-21.

In [ ]:
from __future__ import annotations

import datetime as dt
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('could not locate prod.db walking up from cwd')
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'databases' / 'prod.db'

SESSIONS = {'asia': (0, 7), 'london': (7, 14), 'ny': (14, 21)}
SESSION_COLORS = {'asia': '#cc6633', 'london': '#3344aa', 'ny': '#aa3333'}
print(f'DB: {DB}')

## Load hourly tables

In [ ]:
def _load(table: str, ts_col: str = 'timestamp') -> pd.DataFrame:
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(f'SELECT * FROM {table} ORDER BY {ts_col}', con)
    con.close()
    df['ts'] = pd.to_datetime(df[ts_col], unit='s', utc=True)
    df = df.set_index('ts')
    # Recent backfill left duplicate-timestamp rows; keep the last.
    return df[~df.index.duplicated(keep='last')]

perp = _load('cd_futures_ohlcv')
spot = _load('cd_spot_binance')
oi   = _load('cd_open_interest')
fund = _load('cd_funding_rate')

start = oi.index.min()
print(f'OI starts {start}; trimming all tables to that.')
perp = perp.loc[start:]
spot = spot.loc[start:]
fund = fund.loc[start:]
print(f'perp {len(perp):,}  spot {len(spot):,}  oi {len(oi):,}  funding {len(fund):,}')

## Hourly enriched frame

In [ ]:
hourly = pd.DataFrame(index=perp.index)
hourly['perp_open']  = perp['open']
hourly['perp_high']  = perp['high']
hourly['perp_low']   = perp['low']
hourly['perp_close'] = perp['close']
hourly['perp_vol']   = perp['volume']
hourly['perp_cvd']   = perp['volume_buy'] - perp['volume_sell']
hourly['spot_close'] = spot['close'].reindex(hourly.index)
hourly['spot_cvd']   = (spot['volume_buy'] - spot['volume_sell']).reindex(hourly.index)
hourly['oi_close']   = oi['oi_close'].reindex(hourly.index).ffill(limit=3)
hourly['funding']    = fund['fr_close'].reindex(hourly.index).ffill(limit=8)

def _session_of(h):
    for name, (lo, hi) in SESSIONS.items():
        if lo <= h < hi:
            return name
    return None

hourly['session'] = hourly.index.hour.map(_session_of)
hourly = hourly.dropna(subset=['session', 'perp_close', 'oi_close']).copy()
hourly['date'] = hourly.index.date
print(f'{len(hourly):,} session-tagged hourly bars')

## Per-session aggregates

In [ ]:
def _agg(g: pd.DataFrame) -> pd.Series:
    oi_o, oi_c = g['oi_close'].iloc[0], g['oi_close'].iloc[-1]
    return pd.Series({
        'open':         g['perp_open'].iloc[0],
        'high':         g['perp_high'].max(),
        'low':          g['perp_low'].min(),
        'close':        g['perp_close'].iloc[-1],
        'volume':       g['perp_vol'].sum(),
        'perp_cvd_sum': g['perp_cvd'].sum(),
        'spot_cvd_sum': g['spot_cvd'].sum(),
        'd_oi_pct':     (oi_c - oi_o) / max(oi_o, 1e-9),
        'mean_funding': g['funding'].mean(),
    })

sess = hourly.groupby(['date', 'session']).apply(_agg, include_groups=False)
print(f'{len(sess):,} (date, session) rows')

## Flatten + sweep-bar CVDs

For each UTC day:
- one row with `asia_*`, `london_*`, `ny_*` aggregates
- `day_high`, `day_low` across london ∪ ny
- perp / spot CVD at the post-asia low bar (used by short divergence)
- perp / spot CVD at the post-asia high bar (used by long divergence)

In [ ]:
day = sess.unstack('session')
day.columns = [f'{sn}_{f}' for f, sn in day.columns]
day = day.dropna()
day['asia_range']           = (day['asia_high'] - day['asia_low']).clip(lower=1e-9)
day['day_high']             = day[['london_high', 'ny_high']].max(axis=1)
day['day_low']              = day[['london_low',  'ny_low' ]].min(axis=1)
day['squeeze_peak_session'] = np.where(day['ny_high'] > day['london_high'], 'ny', 'london')
day['dump_peak_session']    = np.where(day['ny_low']  < day['london_low'],  'ny', 'london')

post_asia = hourly[hourly['session'] != 'asia'].copy()
low_bar  = post_asia.loc[post_asia.groupby('date')['perp_low'].idxmin()]
high_bar = post_asia.loc[post_asia.groupby('date')['perp_high'].idxmax()]
day = day.join(low_bar.set_index('date')[['perp_cvd', 'spot_cvd']].rename(
    columns={'perp_cvd': 'sweep_low_perp_cvd', 'spot_cvd': 'sweep_low_spot_cvd'}))
day = day.join(high_bar.set_index('date')[['perp_cvd', 'spot_cvd']].rename(
    columns={'perp_cvd': 'sweep_high_perp_cvd', 'spot_cvd': 'sweep_high_spot_cvd'}))
day = day.dropna(subset=['sweep_low_perp_cvd', 'sweep_high_perp_cvd'])

print(f'{len(day):,} complete days  ({day.index.min()} - {day.index.max()})')

## Scoring — SHORT squeeze (4 legs, 10 checks)

In [ ]:
THRESH = {
    'd_oi_pct_min':  0.005,
    'rip_min_atr':   0.5,
    'reclaim_pct':   0.998,
    'fade_min_atr':  0.3,
    'fade_min_pct':  0.005,
}

def short_score(r):
    g = {
        'oi_up':    r['asia_d_oi_pct']     > THRESH['d_oi_pct_min'],
        'fund_neg': r['asia_mean_funding'] < 0,
        'down':     r['asia_close']        < r['asia_open'],
    }
    rip = (r['day_high'] - r['day_low']) / r['asia_range']
    ss = {
        'low_break':   r['day_low']  <  r['asia_low'],
        'rip':         rip            >= THRESH['rip_min_atr'],
        'reclaim':     r['day_high'] >= r['asia_open'] * THRESH['reclaim_pct'],
        'perp_buying': r['london_perp_cvd_sum'] > 0,
    }
    fade_atr = (r['day_high'] - r['ny_close']) / r['asia_range']
    fade_pct = r['ny_close'] / max(r['day_high'], 1e-9) - 1.0
    f = {
        'atr_fade': fade_atr >= THRESH['fade_min_atr'],
        'pct_fade': fade_pct <= -THRESH['fade_min_pct'],
    }
    c = {'perp_lower_at_sweep': r['sweep_low_perp_cvd'] < r['sweep_low_spot_cvd']}
    return pd.Series({
        's_g':  sum(g.values()),
        's_ss': sum(ss.values()),
        's_f':  sum(f.values()),
        's_c':  sum(c.values()),
        's_total': sum(g.values()) + sum(ss.values()) + sum(f.values()) + sum(c.values()),
    })

## Scoring — LONG squeeze (mirror, 4 legs, 10 checks)

In [ ]:
def long_score(r):
    g = {
        'oi_up':    r['asia_d_oi_pct']     > THRESH['d_oi_pct_min'],
        'fund_pos': r['asia_mean_funding'] > 0,
        'up':       r['asia_close']        > r['asia_open'],
    }
    rip = (r['day_high'] - r['day_low']) / r['asia_range']
    ss = {
        'high_break':   r['day_high'] >  r['asia_high'],
        'rip':          rip            >= THRESH['rip_min_atr'],
        'reclaim':      r['day_low']  <= r['asia_open'] * (2 - THRESH['reclaim_pct']),
        'perp_selling': r['london_perp_cvd_sum'] < 0,
    }
    bounce_atr = (r['ny_close'] - r['day_low']) / r['asia_range']
    bounce_pct = r['ny_close'] / max(r['day_low'], 1e-9) - 1.0
    f = {
        'atr_bounce': bounce_atr >= THRESH['fade_min_atr'],
        'pct_bounce': bounce_pct >= THRESH['fade_min_pct'],
    }
    c = {'perp_higher_at_sweep': r['sweep_high_perp_cvd'] > r['sweep_high_spot_cvd']}
    return pd.Series({
        'l_g':  sum(g.values()),
        'l_ss': sum(ss.values()),
        'l_f':  sum(f.values()),
        'l_c':  sum(c.values()),
        'l_total': sum(g.values()) + sum(ss.values()) + sum(f.values()) + sum(c.values()),
    })

scored = day.join(day.apply(short_score, axis=1)).join(day.apply(long_score, axis=1))
print(f'short max: {scored["s_total"].max()}/10   long max: {scored["l_total"].max()}/10')
print(f'short mean: {scored["s_total"].mean():.2f}   long mean: {scored["l_total"].mean():.2f}')

## Leg-conjunction hit table — short + long + union

In [ ]:
N = len(scored)
print(f'total days: {N}\n')

def hits(prefix, ss_min, f_min, c_min, g_eq=None, ss_eq=None, f_eq=None, c_eq=None):
    cols = [f'{prefix}_g', f'{prefix}_ss', f'{prefix}_f', f'{prefix}_c']
    g = scored[cols[0]]; ss = scored[cols[1]]; f = scored[cols[2]]; c = scored[cols[3]]
    mask = pd.Series(True, index=scored.index)
    if g_eq  is not None: mask &= (g == g_eq)
    else: mask &= (g >= 3)
    if ss_eq is not None: mask &= (ss == ss_eq)
    else: mask &= (ss >= ss_min)
    if f_eq  is not None: mask &= (f == f_eq)
    else: mask &= (f >= f_min)
    if c_eq  is not None: mask &= (c == c_eq)
    else: mask &= (c >= c_min)
    return mask

# Build all the named filters once
ts = hits('s', ss_min=3, f_min=1, c_min=1)                                 # textbook short
tl = hits('l', ss_min=3, f_min=1, c_min=1)                                 # textbook long
cs = hits('s', ss_min=4, f_min=1, c_min=1, ss_eq=4)                        # clean short
cl = hits('l', ss_min=4, f_min=1, c_min=1, ss_eq=4)                        # clean long
ps = hits('s', ss_min=4, f_min=2, c_min=1, ss_eq=4, f_eq=2)                # perfect short
pl = hits('l', ss_min=4, f_min=2, c_min=1, ss_eq=4, f_eq=2)                # perfect long

print(f'{"filter":40s} {"SHORT":>7s} {"LONG":>7s} {"EITHER":>8s} {"either rate":>13s}')
print('-' * 80)
def row(label, sm, lm):
    u = sm | lm
    print(f'{label:40s} {int(sm.sum()):7d} {int(lm.sum()):7d} {int(u.sum()):8d}  {u.mean():11.2%}')
row('textbook  (g==3 & ss>=3 & f>=1 & c==1)', ts, tl)
row('clean     (g==3 & ss==4 & f>=1 & c==1)', cs, cl)
row('perfect   (10/10)',                      ps, pl)

print(f'\ntextbook intersection (both fire): {int((ts & tl).sum())} (legs are mutually exclusive on grind)')
print(f'chop days (both sides >= 7/10):    {int(((scored.s_total >= 7) & (scored.l_total >= 7)).sum())}')

## Year breakdown

In [ ]:
import pandas as _pd
yr = _pd.to_datetime(scored.index).year
ybr = _pd.DataFrame({
    'year': yr,
    'short_textbook': ts.values,
    'long_textbook':  tl.values,
    'perfect_short':  ps.values,
    'perfect_long':   pl.values,
})
agg = ybr.groupby('year').sum()
agg['either_textbook'] = agg['short_textbook'] + agg['long_textbook']
agg

## Case study — 2026-02-11

Should be SHORT 10/10, LONG ≪ 10 (asia grind direction is down, not up).

In [ ]:
target = dt.date(2026, 2, 11)
r = scored.loc[target]
print(f'{target}')
print(f'  SHORT  g={int(r.s_g)}/3  ss={int(r.s_ss)}/4  f={int(r.s_f)}/2  c={int(r.s_c)}/1  -> {int(r.s_total)}/10')
print(f'  LONG   g={int(r.l_g)}/3  ss={int(r.l_ss)}/4  f={int(r.l_f)}/2  c={int(r.l_c)}/1  -> {int(r.l_total)}/10')
print()
print(f'  grind:  asia oi={r.asia_d_oi_pct*100:+.2f}%  fund={r.asia_mean_funding*1e4:+.2f}bp  close<open={r.asia_close<r.asia_open}')
print(f'  ss:     day_low={r.day_low:.0f} (vs asia_low={r.asia_low:.0f}, asia_open={r.asia_open:.0f})')
print(f'          day_high={r.day_high:.0f} (vs asia_open={r.asia_open:.0f}, asia_high={r.asia_high:.0f})')
print(f'          rip = {(r.day_high-r.day_low)/r.asia_range:.2f} x asia_range, squeeze peaked in {r.squeeze_peak_session}')
print(f'  fade:   ny_close={r.ny_close:.0f}  fade from day_high = {(r.day_high-r.ny_close)/r.asia_range:.2f} x asia_range')
print(f'  cvd:    at sweep low bar  perp={r.sweep_low_perp_cvd:+.1f}  spot={r.sweep_low_spot_cvd:+.1f}')
print(f'          at sweep high bar perp={r.sweep_high_perp_cvd:+.1f}  spot={r.sweep_high_spot_cvd:+.1f}')

## Top textbook hits — both sides

Top hits ranked by score, separately for short and long.

In [ ]:
short_hits = scored.loc[ts].sort_values(['s_total','s_ss','s_f'], ascending=False)
long_hits  = scored.loc[tl].sort_values(['l_total','l_ss','l_f'], ascending=False)

print(f'=== top 15 SHORT textbook hits ({int(ts.sum())} total) ===')
cols_s = ['s_g','s_ss','s_f','s_c','s_total','asia_d_oi_pct','asia_mean_funding','squeeze_peak_session']
print(short_hits[cols_s].head(15).round(4).to_string())

print(f'\n=== top 15 LONG textbook hits ({int(tl.sum())} total) ===')
cols_l = ['l_g','l_ss','l_f','l_c','l_total','asia_d_oi_pct','asia_mean_funding','dump_peak_session']
print(long_hits[cols_l].head(15).round(4).to_string())

## Render top-N hits as 4-panel charts

In [ ]:
plt.style.use('dark_background')

def plot_day(date, side='auto'):
    d = pd.Timestamp(date, tz='UTC').normalize()
    sl = hourly.loc[d: d + pd.Timedelta(hours=23, minutes=59)]
    if sl.empty:
        print(f'{date}: no data'); return
    s = scored.loc[date]
    if side == 'auto':
        side = 'SHORT' if s['s_total'] >= s['l_total'] else 'LONG'

    fig, axes = plt.subplots(
        4, 1, figsize=(11, 8.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1.4, 1.4, 1.6]}
    )
    axes[0].plot(sl.index, sl['perp_close'], color='white',  lw=1.1, label='perp')
    axes[0].plot(sl.index, sl['spot_close'], color='orange', lw=1.0, alpha=0.8, label='spot')
    ymax = sl['perp_high'].max()
    for name, (lo, hi) in SESSIONS.items():
        s_start = d + pd.Timedelta(hours=lo)
        s_end   = d + pd.Timedelta(hours=hi)
        axes[0].axvspan(s_start, s_end, alpha=0.10, color=SESSION_COLORS[name])
        axes[0].text(s_start + (s_end - s_start)/2, ymax,
                     name, ha='center', va='bottom', color='gray', fontsize=8)
    if side == 'SHORT':
        title = (f'{date}  [SHORT]  {int(s.s_total)}/10  '
                 f'(g {int(s.s_g)}/3 · ss {int(s.s_ss)}/4 · f {int(s.s_f)}/2 · c {int(s.s_c)}/1)')
    else:
        title = (f'{date}  [LONG]   {int(s.l_total)}/10  '
                 f'(g {int(s.l_g)}/3 · ss {int(s.l_ss)}/4 · f {int(s.l_f)}/2 · c {int(s.l_c)}/1)')
    axes[0].set_title(title)
    axes[0].legend(loc='upper left', fontsize=8); axes[0].set_ylabel('price')

    axes[1].plot(sl.index, sl['oi_close'], color='cyan', lw=1.0); axes[1].set_ylabel('OI')
    axes[2].plot(sl.index, sl['funding'] * 1e4, color='magenta', lw=1.0)
    axes[2].axhline(0, color='gray', lw=0.5); axes[2].set_ylabel('funding (bp)')

    axes[3].plot(sl.index, sl['perp_cvd'].cumsum(), color='red',  lw=1.0, label='perp cvd')
    axes[3].plot(sl.index, sl['spot_cvd'].cumsum(), color='lime', lw=1.0, label='spot cvd')
    axes[3].axhline(0, color='gray', lw=0.5); axes[3].set_ylabel('cum CVD'); axes[3].legend(loc='upper left', fontsize=8)

    plt.tight_layout(); plt.show()

# Render the screenshot (short) and the top 3 of each side
print('--- screenshot day ---')
plot_day(dt.date(2026, 2, 11), side='SHORT')

print('\n--- top 3 SHORT textbook hits ---')
for d in short_hits.index[:3]:
    if d != dt.date(2026, 2, 11):
        plot_day(d, side='SHORT')

print('\n--- top 3 LONG textbook hits ---')
for d in long_hits.index[:3]:
    plot_day(d, side='LONG')

## Next steps

- **Sub-hour resolution** — minute-level perp+spot CVD would let us check
  divergence at the actual sweep candle, not the hourly bar.
- **Sized strategy backtest** — fade the squeeze high / dump low. Wire
  through `services/margin_sim.py`. Long-side hits are 4.6× more common, so
  the long-side fade probably carries more capital efficiency.
- **Volatility-floor variant** — some long-textbook hits are sleepy days
  where small swings clear ATR-normalized thresholds (e.g. 2023-02-13:
  asia_range only 2.4% of asia_open). Adding `asia_range/asia_open ≥ 1.5%`
  would drop long count from 123 to ~70-80 high-quality hits.
- **Cross-day** — let grind→sweep→fade roll across UTC midnight so NY-as-grind
  is allowed.
- **Era split** — 2023 was a long-squeeze hotbed (42 hits) vs 2024 (28). Split
  using era boundaries from [`r4_study/era_split.ipynb`](../r4_study/era_split.ipynb)
  and check whether the pattern's expectancy is era-dependent.
- **Liquidations panel** — once `cd_liquidations` has > 1 year of history
  (only since 2026-02 today).